In [22]:
#!/usr/bin/env python3
"""
fetch_coingecko_hourly_365d.py

Fetches 365 days of **hourly** data for each MVP coin from CoinGecko.
- Endpoint: /coins/{id}/market_chart?vs_currency=usd&days=365&interval=hourly
- Output : historical_python_output.parquet
"""

import logging
import sys
import time

import polars as pl
import requests

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
API_BASE = "https://api.coingecko.com/api/v3"
MVP_COINS = ["bitcoin", "ethereum", "binancecoin", "solana"]
OUTPUT_FILE = "historical_python_output.parquet"

# Polite delay between calls (free tier ≈ 10‑30 requests / minute)
SLEEP_SECONDS = 2.0

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Extraction
# ---------------------------------------------------------------------------
def fetch_hourly_data(session: requests.Session, coin_id: str) -> dict:
    """
    Returns raw JSON for 365 days of hourly data.
    Keys: 'prices', 'market_caps', 'total_volumes'
    """
    url = f"{API_BASE}/coins/{coin_id}/market_chart"
    params = {"vs_currency": "usd", "days": 90, "interval": "hourly"}
    logger.info("Fetching %s (hourly, 365d)", coin_id)
    resp = session.get(url, params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()


# ---------------------------------------------------------------------------
# Transformation
# ---------------------------------------------------------------------------
def raw_to_df(coin_id: str, raw: dict) -> pl.DataFrame:
    """
    Convert the three nested arrays (prices, market_caps, total_volumes)
    into a flat DataFrame with columns:
        coin_id, timestamp_ms, price, market_cap, volume
    """
    def list_to_df(data: list, col_name: str) -> pl.DataFrame:
        return (
            pl.DataFrame(data, schema=["timestamp_ms", col_name], orient="row")
            .with_columns(pl.col("timestamp_ms").cast(pl.Int64))
        )

    df_p = list_to_df(raw["prices"], "price")
    df_m = list_to_df(raw["market_caps"], "market_cap")
    df_v = list_to_df(raw["total_volumes"], "volume")

    # Join on timestamp_ms (all three arrays share identical timestamps)
    df = df_p.join(df_m, on="timestamp_ms").join(df_v, on="timestamp_ms")
    df = df.with_columns(pl.lit(coin_id).alias("coin_id"))
    return df.select(["coin_id", "timestamp_ms", "price", "market_cap", "volume"])


# ---------------------------------------------------------------------------
# Data quality checks
# ---------------------------------------------------------------------------
def validate_dataframe(df: pl.DataFrame) -> None:
    """Raise ValueError on any DQ failure."""
    # Completeness: no nulls in price, market_cap, volume
    for col in ["price", "market_cap", "volume"]:
        null_count = df.select(pl.col(col).null_count()).item()
        if null_count:
            raise ValueError(f"{null_count} nulls found in column '{col}'")

    # Validity: timestamp_ms must be positive integer, price > 0
    bad_ts = df.filter(pl.col("timestamp_ms") <= 0)
    if bad_ts.height:
        raise ValueError(f"{bad_ts.height} rows with non‑positive timestamp_ms")

    bad_price = df.filter(pl.col("price") <= 0)
    if bad_price.height:
        raise ValueError(f"{bad_price.height} rows with price <= 0")

    logger.info("✅ All data quality checks passed.")


# ---------------------------------------------------------------------------
# Main orchestration
# ---------------------------------------------------------------------------
def main():
    all_dfs = []

    with requests.Session() as session:
        for coin in MVP_COINS:
            try:
                raw = fetch_hourly_data(session, coin)
                df = raw_to_df(coin, raw)
                all_dfs.append(df)
                logger.info("  ↳ %d rows for %s", df.height, coin)
            except Exception as e:
                logger.error("Failed to fetch or transform %s: %s", coin, e)
                continue

            time.sleep(SLEEP_SECONDS)

    if not all_dfs:
        logger.error("No data collected. Aborting.")
        sys.exit(1)

    final_df = pl.concat(all_dfs, how="vertical")
    logger.info("Combined DataFrame shape: %s", final_df.shape)

    validate_dataframe(final_df)

    final_df.write_parquet(OUTPUT_FILE)
    logger.info("Saved %d rows to %s", final_df.height, OUTPUT_FILE)


if __name__ == "__main__":
    main()

2026-05-30 14:54:46,648 [INFO] Fetching bitcoin (hourly, 365d)
2026-05-30 14:54:47,285 [INFO]   ↳ 2168 rows for bitcoin
2026-05-30 14:54:49,286 [INFO] Fetching ethereum (hourly, 365d)
2026-05-30 14:54:49,717 [INFO]   ↳ 2167 rows for ethereum
2026-05-30 14:54:51,718 [INFO] Fetching binancecoin (hourly, 365d)
2026-05-30 14:54:52,325 [INFO]   ↳ 2167 rows for binancecoin
2026-05-30 14:54:54,326 [INFO] Fetching solana (hourly, 365d)
2026-05-30 14:54:54,728 [INFO]   ↳ 2167 rows for solana
2026-05-30 14:54:56,730 [INFO] Combined DataFrame shape: (8669, 5)
2026-05-30 14:54:56,731 [INFO] ✅ All data quality checks passed.
2026-05-30 14:54:56,734 [INFO] Saved 8669 rows to historical_python_output.parquet
